## Get started
This tutorial will show you how asynctorch can be used. Make sure asynctorch, snnTorch and tonic have been installed! Tonic is not an automatic dependency of asynctorch because it is not required for all use cases. For CUDA support, make sure you have a working CUDA installation and the correct version of PyTorch installed.

Check our paper for more details: https://arxiv.org/abs/2408.05098

### Import packages

In [ ]:
import torch
import tonic
import matplotlib.pyplot as plt
import tonic.transforms as transforms
from asynctorch.simulator.async_simulator import AsyncSimulator
from asynctorch.nn.architecture.mixed_architecture import MixedArchitecture, AsyncNetwork
from asynctorch.nn.neuron.lif_state import LIFState
from asynctorch.simulator.extensions.spike_dropout_extension import SpikeDropoutExtension
from asynctorch.simulator.spike_scheduler import RandomSpikeScheduler
from asynctorch.simulator.spike_selector import SpikeSelector
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn as nn
import snntorch.functional as SF
from asynctorch.utils.surrogate import ATan
from math import prod
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

### Load data
For this, tonic is used. This can take same time when it is run for the first time.

In [2]:
timestep_size = 10000
transform = transforms.ToFrame(sensor_size=tonic.datasets.NMNIST.sensor_size, time_window=timestep_size)
train_dataset = tonic.datasets.NMNIST(save_to="./data", transform=transform, train=True, first_saccade_only=True)
test_dataset = tonic.datasets.NMNIST(save_to="./data", transform=transform, train=False, first_saccade_only=True)
collate_fn = tonic.collation.PadTensors(batch_first=False)

### Define the network and parameters for simulator
We provide two network architectures. An architecture consists of just the connecting layer, so not the (spiking) activations / neurons. You can use the layers from PyTorch, like nn.Conv2d and nn.Linear. On top of that, you must also provide their input shape. 
In a later cell, these definitions will be used to construct the network. We assume here that you want a fully sequential network. For more advanced structures like residual connections, you must manually connect each layer to neurons. This will be discussed in another tutorial (to be added later). For now, after running "CNN" or "Linear", just run the "Build" cell to construct.

#### CNN

In [3]:
n_outputs = 10
input_shape = (2, 34, 34) # Shape of the input data
layer1_shape = (8, 16, 16)
layer2_shape = (16, 8, 8)
layer3_shape = (32, 4, 4)
module_per_layer = [nn.Conv2d(2, 8, kernel_size=3, stride=2, padding=0, bias=False), 
                    nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1, bias=False), 
                    nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1, bias=False),
                    nn.Linear(prod(layer3_shape), n_outputs, bias=False)]
shape_per_layer = [input_shape, layer1_shape, layer2_shape, (prod(layer3_shape), ), (n_outputs, )]
neurons_per_layer = [prod(shape) for shape in shape_per_layer[1:]]
n_neurons = sum(neurons_per_layer)

#### Linear

In [3]:
n_outputs = 10
input_shape = (2*34*34, ) # Shape of the input data, we flatten the input data in this way
layer1_shape = (64, )
layer2_shape = (64, )
layer3_shape = (64, )
module_per_layer = [nn.Linear(prod(input_shape), prod(layer1_shape), bias=False),
                    nn.Linear(prod(layer1_shape), prod(layer2_shape), bias=False),
                    nn.Linear(prod(layer2_shape), prod(layer3_shape), bias=False),
                    nn.Linear(prod(layer3_shape), n_outputs, bias=False)]
shape_per_layer = [(prod(input_shape), ), layer1_shape, layer2_shape, layer3_shape, (n_outputs, )]
neurons_per_layer = [prod(shape) for shape in shape_per_layer[1:]]
n_neurons = sum(neurons_per_layer)

#### Build
The building process consists of first taking the network definitions from the previous cells and then using those to construct an architecture module. This will provide the logic for how to forward spikes and currents. How the neurons behave is defined in a state module, in this case LIF neurons are used. The last step is defining how spikes are selected and then combining everything into the simulator environment. Optionally, "extensions" can be provided. In this example, spike dropout will be used for regularization, but only applied to the input. Different from torch dropout, it is not directly included in the architecture, but rather added as an extension.

In [ ]:
# Network architecture
async_network = AsyncNetwork.build_sequential(module_per_layer, shape_per_layer).to(device)
print(async_network)
print('Number of neurons:', n_neurons)
print('Number of parameters:', sum(p.numel() for p in async_network.parameters() if p.requires_grad))
network_module = MixedArchitecture(async_network, device)

# Neuron model
spike_grad = ATan()
state_module = LIFState(
    neurons_per_layer,
    tau_m=1000,
    membrane_threshold=0.3,
    spike_grad=spike_grad,
    device=device,
    refrac_dropout=0.8, # Probability that a neuron can spike again in the same forward pass during training, check paper for more details
)

# Spike selection
spike_scheduler = RandomSpikeScheduler(state_module)
spike_selector_module = SpikeSelector(
    network_module,
    spike_scheduler,
    forward_group_size=128, # How many spikes are processed in parallel per forward step
    device=device,
    prioritize_input=True,
    log_queue_length=True
)
# Simulator and extensions
input_dropout = SpikeDropoutExtension(p=0.25, apply_to_input=True, apply_to_network=False)
async_simulator = AsyncSimulator(state_module, spike_selector_module, forward_step_extensions=[input_dropout])

## Start training
For training, we use the standard PyTorch training loop with some snnTorch specific functions. Instead of calling a forward pass on the network, we call the forward function of the simulator. This function takes the input data and the time step, and returns the output spikes of the network. The output spikes can then be used to calculate the loss and update the weights of the network.

In [ ]:

train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)
async_simulator.train()
n_epochs = 1
optimizer = torch.optim.Adam(async_simulator.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = SF.ce_rate_loss()
accuracy_function = SF.accuracy_rate
losses = []
accuracies = []
for epoch in range(n_epochs):
    batch_iter = iter(train_dataloader)
    for data, targets in tqdm(batch_iter):
        data = data.to(device)
        targets = targets.to(device)
        async_simulator.reset_state() # Reset the state of the simulator (including the state of the neurons)

        # Forward pass
        ys = []
        for t in range(data.shape[0]):
            ts_data = data[t].view(data[t].shape[0], -1)
            spk_out = async_simulator(ts_data, dt=timestep_size)[:, n_neurons-n_outputs:]
            ys.append(spk_out) 
        y = torch.stack(ys)
        loss = loss_fn(y, targets)
        losses.append(loss.item())
        accuracy = accuracy_function(y, targets)
        accuracies.append(accuracy)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

### Plot the training progress

In [ ]:
fig, ax1 = plt.subplots()

ax2 = ax1.twinx()
ax1.plot(losses, color='C0')
ax2.plot(accuracies, color='C1')

ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss', color='C0')
ax2.set_ylabel('Accuracy', color='C1')

plt.show()

## Test the model

In [ ]:
async_simulator.eval()
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)
accuracies = []
batch_sizes = []

with torch.inference_mode():
    for data, targets in tqdm(test_dataloader):
        data = data.to(device)
        targets = targets.to(device)
        async_simulator.reset_state()
        ys = []
        for t in range(data.shape[0]):
            ts_data = data[t].float().view(data[t].shape[0], -1)
            spk_out = async_simulator(ts_data, dt=timestep_size)[:, n_neurons-n_outputs:]
            ys.append(spk_out) 
        y = torch.stack(ys).sum(dim=0)
        predictions = torch.argmax(y, dim=1)
        accuracy = predictions.eq(targets).float().mean().cpu().item()
        accuracies.append(accuracy)
        batch_sizes.append(data.shape[0])

print("Test accuracy: ", sum([a*b for a, b in zip(accuracies, batch_sizes)]) / sum(batch_sizes))